# 02 · Train (checkpoint every epoch, automatic resume)

Runtime: **GPU** (Runtime → Change runtime type → T4 GPU).

Each run lives in `DRIVE_ROOT/runs/<RUN_NAME>/`: `config.json`, `metrics.csv`, `last.pt` (resume point, every epoch), `best.pt` (best val ROC-AUC).

* **Disconnected or stopped?** Run the notebook again with the same `RUN_NAME` → it continues after the last finished epoch.
* **Train longer?** Raise `EPOCHS` → continues from where it stopped.
* **Different model, image size or clip length?** New `RUN_NAME` (the code refuses to mix them).
* **Leave-one-method-out (LOMO):** set `EXCLUDE_METHOD` (e.g. `NeuralTextures`) and a new `RUN_NAME` such as `mnv2_lomo_NT`.
  That method is removed from training *and* validation, so neither the model nor its threshold ever sees it;
  notebook 03 then measures how well this unseen manipulation is still caught.

| Model | What | Typical setting |
|---|---|---|
| `b2_cnn` | Baseline B2: small CNN from scratch, one frame | 128 px, lr 1e-3, 10 epochs |
| `mnv2` | MobileNetV2 (ImageNet), one frame | 224 px, lr 3e-4, 1 frozen epoch, 8 epochs |
| `mnv2_gru` | MobileNetV2 per frame + GRU over 16 frames | 224 px, lr 1e-4, 1 frozen epoch, 6 epochs |

In [ ]:
#@title 0 · Setup (run after every restart)
DRIVE_ROOT = "/content/drive/MyDrive/IS665_Deepfake"  #@param {type:"string"}
REPO_URL = "https://github.com/payelsarkarr/is665-team6-deepfake-recognition"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
REPO = "/content/repo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)       # newest code from GitHub
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, f"{REPO}/src")
import cv2
if not hasattr(cv2, "CascadeClassifier"):
    !pip -q install "opencv-python-headless<5"
    print("OpenCV 4 installed -> Runtime > Restart session, then run this cell again.")
META = "/content/FaceForensics/dataset"            # official split files
if not os.path.exists(META):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ondyari/FaceForensics.git", "/content/FaceForensics"], check=True)
import importlib, ffpp
for m in ["data", "crops", "dataset", "models", "metrics", "train", "evaluate", "download"]:
    importlib.reload(importlib.import_module(f"ffpp.{m}"))
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 1 · Settings of this run
CACHE_VERSION = "crops_v1"  #@param {type:"string"}
RUN_NAME = "b2_cnn_v1"  #@param {type:"string"}
MODEL = "b2_cnn"  #@param ["b2_cnn", "mnv2", "mnv2_gru"]
IMG_SIZE = 128  #@param {type:"integer"}
EPOCHS = 10  #@param {type:"integer"}
LR = 1e-3  #@param {type:"number"}
BATCH_SIZE = 64  #@param {type:"integer"}
FREEZE_BACKBONE_EPOCHS = 0  #@param {type:"integer"}
CLIP_LEN = 16  #@param {type:"integer"}
#@markdown Random training samples per video and epoch (frames for b2_cnn/mnv2, clips for mnv2_gru):
SAMPLES_PER_VIDEO = 4  #@param {type:"integer"}
#@markdown Stop after this many epochs without improvement (0 = off):
PATIENCE = 3  #@param {type:"integer"}
#@markdown Leave-one-method-out: method hidden from training and validation ("none" = normal run):
EXCLUDE_METHOD = "none"  #@param ["none", "Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
cfg = dict(run_name=RUN_NAME, model=MODEL, img_size=IMG_SIZE, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE,
           freeze_backbone_epochs=FREEZE_BACKBONE_EPOCHS, clip_len=CLIP_LEN, samples_per_video=SAMPLES_PER_VIDEO,
           patience=PATIENCE, pretrained=True, num_workers=2,
           exclude_methods=[] if EXCLUDE_METHOD == "none" else [EXCLUDE_METHOD])
RUNS_DIR = f"{DRIVE_ROOT}/runs"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none – switch the runtime to GPU")

In [ ]:
#@title 2 · Copy the clip cache from Drive to the runtime (faster than reading from Drive)
from ffpp.dataset import copy_cache_local, load_index
LOCAL_CACHE = copy_cache_local(f"{DRIVE_ROOT}/cache/{CACHE_VERSION}", f"/content/cache/{CACHE_VERSION}")
for s in ["train", "val", "test"]:
    d = load_index(LOCAL_CACHE, s)
    print(f"{s:5s}: {len(d)} videos ({int((d.label == 0).sum())} real, {int((d.label == 1).sum())} fake)")

In [ ]:
#@title 3 · Train (resumes automatically)
from ffpp.train import train
hist = train(cfg, LOCAL_CACHE, RUNS_DIR)
hist

In [ ]:
#@title 4 · Learning curves
import matplotlib.pyplot as plt, pandas as pd
h = pd.read_csv(f"{RUNS_DIR}/{RUN_NAME}/metrics.csv")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(h.epoch, h.train_loss, marker="o"); ax[0].set_title("Training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(h.epoch, h.val_roc_auc, marker="o", label="val ROC-AUC")
ax[1].plot(h.epoch, h.val_recall_at_fpr, marker="o", label="val recall at FPR ≤ 0.2")
ax[1].set_ylim(0, 1); ax[1].set_xlabel("epoch"); ax[1].legend(frameon=False); ax[1].set_title("Validation (per video)")
for a in ax: a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()